# BE-HPG — Phase 4: Main comparison (Spleen)

Trains + evaluates **ProtoNet, Meta-reweight, BE-HPG** on identical episodes and reports
Dice / IoU / HD95 / ASD at 1-shot and 5-shot.

**Run order**
1. **Runtime ▸ GPU (T4)**. Upload `be-hpg-src.zip` when cell 3 asks (set `FORCE_UPLOAD=True`
   first if you changed the code since last time — **you did: the loss was updated**).
2. Run with **`SMOKE = True`** once (~2 min) to confirm all three models predict non-empty
   masks (the meta-reweight fix). Then set **`SMOKE = False`** for the full run (~30–45 min,
   resumable; checkpoints every 250 episodes to Drive).

In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/be-hpg'
print('Drive:', DRIVE_ROOT)

In [ ]:
# 2) Dependencies
!pip -q install timm medpy

In [ ]:
# 3) Get BE-HPG code (upload once -> cached on Drive). Set FORCE_UPLOAD=True after code changes.
import os, sys, shutil
FORCE_UPLOAD = True        # <-- the loss code changed since Phase 3; re-upload the new zip once
zip_drive = f'{DRIVE_ROOT}/be-hpg-src.zip'
if FORCE_UPLOAD or not os.path.exists(zip_drive):
    from google.colab import files
    print('Upload the NEW be-hpg-src.zip:')
    up = files.upload()
    with open(zip_drive, 'wb') as f:
        f.write(up[list(up)[0]])
code_dir = '/content/be-hpg'
shutil.rmtree(code_dir, ignore_errors=True)
os.makedirs(code_dir, exist_ok=True)
os.system(f'unzip -q -o "{zip_drive}" -d "{code_dir}"')
if code_dir not in sys.path:
    sys.path.insert(0, code_dir)
for _m in [k for k in sys.modules if k == 'src' or k.startswith('src.')]:
    del sys.modules[_m]        # drop cached modules so a re-uploaded zip takes effect (no restart needed)
import inspect
from src.models.be_hpg import BEHPG
print('code ready | new BE-HPG loaded =', 'use_ssp' in inspect.signature(BEHPG.__init__).parameters)

In [ ]:
# 4) CONFIG  --  SMOKE=True to verify, then SMOKE=False for the full run
import torch
SMOKE = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
suffix = '_smoke' if SMOKE else ''
NPZ = f'{DRIVE_ROOT}/data/spleen/spleen{suffix}.npz'
MANIFEST = f'{DRIVE_ROOT}/data/spleen/manifest{suffix}.csv'
TRAIN_EP = 50 if SMOKE else 1500
EVAL_EP = 10 if SMOKE else 150
EPISODE_SEED = 1234        # fixes the SAME eval episodes across all three models (fairness)
MODELS = {'protonet': 'protonet.yaml', 'meta_reweight': 'meta_reweight.yaml', 'be_hpg': 'be_hpg.yaml'}
print('device:', DEVICE, '| SMOKE', SMOKE, '| train', TRAIN_EP, 'eval', EVAL_EP)

In [ ]:
# 5) Load data + dataset stats (these numbers go in the paper)
import pandas as pd, numpy as np
mdf = pd.read_csv(MANIFEST) if os.path.exists(MANIFEST) else None
data = np.load(NPZ)
print('total slices:', len(data['images']), '| volumes:', len(set(data['vol_ids'].astype(str))))
if mdf is not None and 'split' in mdf.columns:
    print('\nvolumes per split:'); print(mdf.groupby('split')['vol_id'].nunique())
    print('\nslices per split:'); print(mdf['split'].value_counts())
TRAIN_SPLIT = None if SMOKE else 'train'
TEST_SPLIT = None if SMOKE else 'test'

In [ ]:
# 6) Imports
import time, json
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from src.utils.config import load_config
from src.utils.seed import set_seed
from src.engine.build import build_model
from src.engine.train import train_episodes
from src.engine.eval import eval_episodes
from src.engine.checkpoint import load_checkpoint
from src.data.sampler import sampler_from_npz
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)

In [ ]:
# 7) Train + evaluate the three models (resumable)
all_results = {}
for name, yml in MODELS.items():
    cfg = load_config(f'{code_dir}/configs/{yml}')
    set_seed(cfg.get('seed', 42))
    print('=' * 64); print('MODEL', name, '| backbone', cfg['model']['backbone'],
                           '| lambda', cfg['loss']['boundary_lambda'])
    model = build_model(cfg)
    tr = sampler_from_npz(NPZ, mdf, TRAIN_SPLIT, k_shots=cfg['episode']['k_shot_train'],
                          query_size=cfg['episode']['query_size'], seed=cfg.get('seed', 42))
    opt = AdamW(model.parameters(), lr=cfg['train']['lr'], weight_decay=cfg['train']['weight_decay'])
    sched = CosineAnnealingLR(opt, T_max=TRAIN_EP)
    ckpt = f"{DRIVE_ROOT}/checkpoints/{name}_spleen{suffix}.pt"
    start = 0
    if (not SMOKE) and os.path.exists(ckpt):
        start, _ = load_checkpoint(ckpt, model, opt, sched, map_location=DEVICE)
        print('resumed from episode', start)
    t0 = time.time()
    if start < TRAIN_EP:
        hist, _, _ = train_episodes(model, tr, episodes=TRAIN_EP, start_episode=start, optimizer=opt,
                                    scheduler=sched, lam=cfg['loss']['boundary_lambda'], device=DEVICE,
                                    amp=cfg.get('amp', True), ckpt_path=ckpt, ckpt_every=250,
                                    log_every=max(TRAIN_EP // 10, 1),
                                    balanced=cfg['loss'].get('balanced_bce', True),
                                    max_pw=cfg['loss'].get('bce_max_pos_weight', 20.0))
        print(f'  trained {TRAIN_EP - start} ep in {time.time() - t0:.0f}s | last loss {hist[-1]["total"]:.4f}')
    res = {'model': name, 'backbone': cfg['model']['backbone'], 'lambda': cfg['loss']['boundary_lambda'],
           'train_episodes': TRAIN_EP, 'eval_episodes': EVAL_EP, 'by_shot': {}}
    for k in (1, 5):
        ev = sampler_from_npz(NPZ, mdf, TEST_SPLIT, k_shots=[k],
                              query_size=cfg['episode']['query_size'], seed=EPISODE_SEED)
        res['by_shot'][f'{k}shot'] = eval_episodes(model, ev, episodes=EVAL_EP, k_shot=k, device=DEVICE)
        print(f'  {k}-shot:', res['by_shot'][f'{k}shot'])
    json.dump(res, open(f'{DRIVE_ROOT}/results/{name}_spleen{suffix}.json', 'w'), indent=2)
    all_results[name] = res

print('\n==== DICE SUMMARY ====')
for n, r in all_results.items():
    print(f"  {n:14s} 1shot {r['by_shot']['1shot']['dice']:.3f} | 5shot {r['by_shot']['5shot']['dice']:.3f}")

In [ ]:
# 8) Qualitative figures (query | GT | pred) -> Drive; bring the good ones into paper/figures/
import matplotlib.pyplot as plt
for name, yml in MODELS.items():
    cfg = load_config(f'{code_dir}/configs/{yml}'); set_seed(cfg.get('seed', 42))
    model = build_model(cfg).to(DEVICE).eval()
    ckpt = f"{DRIVE_ROOT}/checkpoints/{name}_spleen{suffix}.pt"
    if (not SMOKE) and os.path.exists(ckpt):
        load_checkpoint(ckpt, model, map_location=DEVICE)
    ev = sampler_from_npz(NPZ, mdf, TEST_SPLIT, k_shots=[1], query_size=1, seed=7)
    rows = 4
    fig, axes = plt.subplots(rows, 3, figsize=(7, 2.3 * rows))
    for r in range(rows):
        b = ev.sample(k=1)
        with torch.no_grad():
            pr = torch.sigmoid(model(b['sup_img'].to(DEVICE), b['sup_mask'].to(DEVICE),
                                     b['qry_img'].to(DEVICE)))
        img = b['qry_img'][0, 0].cpu().numpy()
        gt = b['qry_mask'][0, 0].cpu().numpy()
        pd_ = (pr[0, 0].cpu().numpy() > 0.5)
        for c, (title, ov) in enumerate([('query', None), ('ground truth', gt), ('prediction', pd_)]):
            axes[r, c].imshow(img, cmap='gray')
            if ov is not None:
                axes[r, c].imshow(np.ma.masked_where(ov == 0, ov), cmap='autumn', alpha=0.5)
            if r == 0:
                axes[r, c].set_title(title)
            axes[r, c].axis('off')
    fig.suptitle(name)
    plt.tight_layout()
    out = f'{DRIVE_ROOT}/results/{name}_spleen{suffix}_qualitative.png'
    plt.savefig(out, dpi=140, bbox_inches='tight'); plt.show()
    print('saved', out)

## ✅ Success / report back
- **SMOKE run:** confirm **meta_reweight is no longer ~0** (non-empty masks, surface metrics
  not 100% excluded). All three should print finite losses + metrics. Then set `SMOKE = False`.
- **FULL run:** ~30–45 min; resumable (re-run the notebook if disconnected — it picks up from
  the last checkpoint).

**Bring back to me:**
- The printed output (dataset stats + the DICE SUMMARY + per-model metric dicts).
- Download the 3 `results/<model>_spleen.json` into the repo `results/` folder.
- Download the 3 `results/<model>_spleen_qualitative.png` (I'll select the best for the paper).
- If anything errors, paste the failing cell's output.